# CodonTransformer csi_top10_hc validation three-way evaluation

Read-only comparison of real CDS, the official pretrained baseline, and the formal v1 checkpoint on all 531 validation records. It reuses the frozen refined-v2 rules, codon reference, and test-derived length boundaries. No training is performed.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU is required"
print(torch.cuda.get_device_name(0))

## Clone the private project securely and pin upstream
Store a fine-grained read-only token as the Colab secret `GITHUB_TOKEN`. It is passed only through `GIT_ASKPASS` and is never printed.

In [ ]:
import hashlib
import json
import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata

REPO_URL = "https://github.com/Yuano0o/codontransformer-nb.git"
PROJECT_DIR = Path("/content/codontransformer-project")
UPSTREAM_URL = "https://github.com/Adibvafa/CodonTransformer.git"
UPSTREAM_COMMIT = "4a447b01dab860feb81b647ff1ff88ad598517f4"
HF_MODEL_ID = "adibvafa/CodonTransformer"
HF_MODEL_REVISION = "9744dcc920d813066391fc828d7a590207f148e8"
github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab secret GITHUB_TOKEN is unavailable")
with tempfile.TemporaryDirectory(prefix="git-askpass-") as askpass_dir:
    askpass = Path(askpass_dir) / "askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in\n'
        '  *sername*) printf \'%s\\n\' \'x-access-token\' ;;\n'
        '  *assword*) printf \'%s\\n\' "$COLAB_GITHUB_TOKEN" ;;\n'
        '  *) exit 1 ;;\nesac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "COLAB_GITHUB_TOKEN": github_token})
    if not (PROJECT_DIR / ".git").is_dir():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True, env=env)
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True, env=env)
    env.pop("COLAB_GITHUB_TOKEN", None)
github_token = None
os.chdir(PROJECT_DIR)
UPSTREAM_DIR = PROJECT_DIR / "upstream"
if not (UPSTREAM_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", UPSTREAM_COMMIT], check=True)

## Install dependencies and mount Google Drive

In [ ]:
subprocess.run([os.sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, (str(UPSTREAM_DIR), os.environ.get("PYTHONPATH", ""))))
if str(UPSTREAM_DIR) not in os.sys.path:
    os.sys.path.insert(0, str(UPSTREAM_DIR))
from google.colab import drive
drive.mount("/content/drive")

## Load the frozen refined-v2 configuration and verify immutable inputs

The validation JSONL, codon reference, and completed formal checkpoint remain read-only. Outputs use a separate Drive directory from the test evaluation.

In [ ]:
import yaml
CONFIG_PATH = PROJECT_DIR / "configs/evaluate_validation_refined_v2.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
DRIVE_ROOT = Path("/content/drive/MyDrive/CodonTransformer")
VALIDATION_PATH = DRIVE_ROOT / CONFIG["paths"]["validation_dataset"]
REFERENCE_PATH = DRIVE_ROOT / CONFIG["paths"]["codon_reference"]
RUN_DIR = DRIVE_ROOT / CONFIG["paths"]["formal_run"]
CHECKPOINT = RUN_DIR / "checkpoints" / CONFIG["inputs"]["checkpoint_filename"]
OUTPUT_DIR = RUN_DIR / CONFIG["paths"]["output_directory"]
REFINED_DIR = OUTPUT_DIR / CONFIG["paths"]["refined_output_directory"]

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for label, path in (("validation", VALIDATION_PATH), ("reference", REFERENCE_PATH), ("checkpoint", CHECKPOINT)):
    assert path.is_file(), f"Missing {label}: {path}"
with VALIDATION_PATH.open(encoding="utf-8") as handle:
    assert sum(bool(line.strip()) for line in handle) == CONFIG["inputs"]["expected_records"]
assert sha256(VALIDATION_PATH) == CONFIG["inputs"]["dataset_sha256"]
assert sha256(REFERENCE_PATH) == CONFIG["inputs"]["reference_sha256"]
assert CHECKPOINT.stat().st_size == CONFIG["inputs"]["checkpoint_size_bytes"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent validation output:", OUTPUT_DIR)

## Download and verify the exact official pretrained baseline

In [ ]:
MODEL_DIR = PROJECT_DIR / "models/pretrained"
subprocess.run([
    os.sys.executable, "scripts/download_pretrained.py",
    "--repo-id", HF_MODEL_ID, "--revision", HF_MODEL_REVISION,
    "--output-dir", str(MODEL_DIR),
], check=True)
assert sha256(MODEL_DIR / "model.safetensors") == CONFIG["inputs"]["pretrained_model_safetensors_sha256"]

## Generate or resume the 531-record three-way evaluation

Prediction caches are flushed to Drive every 25 records. Re-running resumes missing baseline or fine-tuned predictions and never trains a model.

In [ ]:
subprocess.run([
    os.sys.executable, "scripts/evaluate_biological_fidelity.py",
    "--model-dir", str(MODEL_DIR),
    "--checkpoint", str(CHECKPOINT),
    "--dataset", str(VALIDATION_PATH),
    "--dataset-role", "validation",
    "--reference-json", str(REFERENCE_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--device", CONFIG["runtime"]["device"],
    "--organism", CONFIG["decoding"]["organism"],
    "--expected-records", str(CONFIG["inputs"]["expected_records"]),
    "--expected-dataset-sha256", CONFIG["inputs"]["dataset_sha256"],
    "--expected-reference-sha256", CONFIG["inputs"]["reference_sha256"],
    "--expected-pretrained-sha256", CONFIG["inputs"]["pretrained_model_safetensors_sha256"],
    "--length-short-max", str(CONFIG["length_boundaries"]["short_max_aa"]),
    "--length-medium-max", str(CONFIG["length_boundaries"]["medium_max_aa"]),
    "--rare-threshold", str(CONFIG["decoding"]["rare_codon_threshold"]),
    "--bootstrap-samples", str(CONFIG["statistics"]["bootstrap_samples"]),
    "--seed", str(CONFIG["statistics"]["seed"]),
    "--flush-every", str(CONFIG["runtime"]["prediction_cache_flush_every"]),
], check=True)

## Apply the frozen refined-v2 decision rules

This step is CPU-only analysis of the completed prediction CSV. It verifies that every stored length bin matches the frozen test-derived boundaries.

In [ ]:
subprocess.run([
    os.sys.executable, "scripts/refine_biological_evaluation.py",
    "--per-sequence-csv", str(OUTPUT_DIR / "per_sequence_metrics.csv"),
    "--reference-json", str(REFERENCE_PATH),
    "--output-dir", str(REFINED_DIR),
    "--dataset-role", "validation",
    "--expected-records", str(CONFIG["inputs"]["expected_records"]),
    "--length-short-max", str(CONFIG["length_boundaries"]["short_max_aa"]),
    "--length-medium-max", str(CONFIG["length_boundaries"]["medium_max_aa"]),
    "--bootstrap-samples", str(CONFIG["statistics"]["bootstrap_samples"]),
    "--seed", str(CONFIG["statistics"]["seed"]),
    "--force",
], check=True)

## Inspect the frozen-rule validation decision and persistent files

In [ ]:
summary_path = REFINED_DIR / "refined_biological_evaluation_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["analysis_version"] == "2.0.0"
assert summary["dataset_role"] == "validation"
assert summary["records"] == 531
print(json.dumps(summary["decision"], indent=2))
print((REFINED_DIR / "refined_biological_evaluation_report.md").read_text())
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size)